In [85]:
import pandas as pd
import numpy as np
import networkx as nx
import re
from pathlib import Path
from etc.parse_ids import XMLParser

# Get the notebook's directory and go up to project root
notebook_dir = Path().resolve()
project_root = notebook_dir.parent
data_folder = project_root / "data" / "resources"
Pilot = data_folder / "PilotStudy_All"

# Get graph prevouly fitered
graph_gml = data_folder / "generated" / "modified_graph.gml"
# Get the original graph for matching ids
human1_xml = data_folder / "Human-GEM.xml"

In [86]:
# Auxiliar function to extract the chebi number from a string like "CHEBI:12345"
def extract_single_chebi(value):
    """Extract CHEBI number from a single CHEBI string"""
    if pd.isna(value):
        return None
    match = re.search(r"CHEBI:(\d+)", str(value))
    if match:
        return int(match.group(1))
    return None

In [87]:
# Load the data file as a pandas DataFrame
data_file = pd.read_excel(Pilot / "Daniel_Suplementary_info.xlsx", sheet_name="Sheet1")

# Filter to only include rows where ID_level is 1
data_file = data_file[data_file.ID_level==1]

# Add new column with extracted CHEBI numbers
data_file['CHEBI_number'] = data_file.CHEBI_ID_Step2.apply(extract_single_chebi)

In [88]:
parser = XMLParser(human1_xml)
df = parser.extract_data()
df_human1 = parser.to_identifier_df()
df_human1['Consortium'] = None

In [89]:
# Convert df_human1 CHEBI to integers for matching
df_human1['chebi_int'] = pd.to_numeric(df_human1['chebi'], errors='coerce').astype('Int64')
# Get set of CHEBIs from data_file
data_file_chebis = set(data_file['CHEBI_number'].dropna().values)
data_file_metanetx = set(data_file['metanetx'].dropna().values)
data_file_keggid = set(data_file['KEGG_ID'].dropna().values)

# Fill Consortium column with boolean: True if CHEBI matches, False otherwise
df_human1['Consortium'] = df_human1['chebi_int'].isin(data_file_chebis)

print(f"Consortium column filled with boolean values")
print(f"True matches: {df_human1['Consortium'].sum()}")
print(f"False (no match): {(~df_human1['Consortium']).sum()}")

Consortium column filled with boolean values
True matches: 356
False (no match): 8100


In [90]:
# Find CHEBIs in data_file that are NOT in df_human1
df_human1_chebis = set(df_human1['chebi_int'].dropna().values)
data_file_chebis_set = set(data_file['CHEBI_number'].dropna().values)

# CHEBIs not found in df_human1
unmatched_chebis = data_file_chebis_set - df_human1_chebis

print(f"Total CHEBIs in data_file: {len(data_file_chebis_set)}")
print(f"CHEBIs found in df_human1: {len(data_file_chebis_set - unmatched_chebis)}")
print(f"CHEBIs NOT found in df_human1: {len(unmatched_chebis)}")
print(f"\nUnmatched CHEBIs: {sorted(unmatched_chebis)}")

Total CHEBIs in data_file: 228
CHEBIs found in df_human1: 105
CHEBIs NOT found in df_human1: 123

Unmatched CHEBIs: [np.int64(4139), np.int64(4828), np.int64(9008), np.int64(11901), np.int64(14314), np.int64(15620), np.int64(15761), np.int64(16020), np.int64(16113), np.int64(16119), np.int64(16231), np.int64(16312), np.int64(16347), np.int64(16373), np.int64(16831), np.int64(17012), np.int64(17016), np.int64(17072), np.int64(17234), np.int64(17597), np.int64(17687), np.int64(17780), np.int64(18123), np.int64(19030), np.int64(19062), np.int64(19065), np.int64(19660), np.int64(21264), np.int64(21553), np.int64(21563), np.int64(21756), np.int64(21949), np.int64(25858), np.int64(27410), np.int64(27596), np.int64(27732), np.int64(27747), np.int64(27838), np.int64(28123), np.int64(28177), np.int64(28238), np.int64(28393), np.int64(28664), np.int64(28717), np.int64(28775), np.int64(28821), np.int64(28871), np.int64(28946), np.int64(30753), np.int64(30776), np.int64(30845), np.int64(35280), np

In [84]:
# Fill Consortium column with boolean: True if metanetx matches, False otherwise
df_human1['Consortium'] = df_human1['kegg'].isin(data_file_keggid)

print(f"True matches: {df_human1['Consortium'].sum()}")
print(f"False (no match): {(~df_human1['Consortium']).sum()}")

True matches: 406
False (no match): 8050


In [81]:
data_file

,ID,Class,Sub class,Pubchem CID,CHEBI_ID,HMDB_ID,LM_ID,KEGG_ID,INCHI_KEY,REFMET_ID,...,Plasma,Mitra,Capitainer,Whatman,Lab,Technique,ID_level,ID level curation,HUMAN1_ID,CHEBI_number
0,(2E)-octenoylcarnitine,Organic acids,Phosphoethanolamines,-,-,-,-,-,-,-,...,Yes,Yes,Yes,Yes,HMGU,HILIC+,1,1 - MS2 Match,None,73037
6,12-Ketochenodeoxycholic acid,Sterol Lipids,C24 bile acids,94235,16312,HMDB0000400,LMST04010176,NaN,MIHNUBCEFJLAGN-DMMBONCOSA-N,RM0128421,...,Yes,No,No,No,HMGU,RPLC+,1,1 - MS2 Match,None,16312
8,1-Azaniumylcyclohexane-1-carboxylate,Organic acids,Amino acids,1366,86534,HMDB0243822,NaN,NaN,WOXWUZCRWJWTRT-UHFFFAOYSA-N,RM0039450,...,Yes,Yes,Yes,Yes,CEMBIO,CE+,1,1 - MS1 Match,None,86534
9,1H-Indole-3-carboxaldehyde,Alkaloids,Simple indole alkaloids,10256,28238,HMDB0029737,NaN,C08493,OLNJUISKUQQNIM-UHFFFAOYSA-N,RM0170863,...,No,Yes,Yes,Yes,AFEKTA,RP+,1,1 - MS2 Match,None,28238
10,1-Methyl nicotinamide,Alkaloids,Nicotinic acid alkaloids,457,16797,HMDB0000699,NaN,C02918,LDHMAVIPBRSVRG-UHFFFAOYSA-O,RM0030166,...,Yes,Yes,Yes,Yes,ICL,HILIC +,1,1 - MS1 Match,MAM00536c,16797
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
353,Urea,Ureas,Ureas,1176,16199,HMDB0000294,NaN,C00086,XSQUKJJJFZCRTK-UHFFFAOYSA-N,RM0117395,...,Yes,Yes,Yes,Yes,"AUTh, CEMBIO","CE+, GC",1,"1 - MS1 Match, 2 - MS2 Match",MAM03121c,16199
354,Uric acid,Nucleic acids,Xanthines,1175,17775,HMDB0000289,NaN,C00366,LEHOTFFKMJEONL-UHFFFAOYSA-N,RM0160856,...,Yes,Yes,Yes,Yes,"AFEKTA, AUTh, CEMBIO, HMGU","CE+, CE-, HILIC+, HILIC-, RP+",1,"1 - MS1 Match, 2 - MS2 Match",MAM03120c,17775
355,Urocanic acid,Alkaloids,Imidazole alkaloids,736715,30817,HMDB0000301,NaN,C00785,LOIYMIARKYCTBW-OWOJBTEDSA-N,RM0135935,...,Yes,Yes,Yes,Yes,ICL,HILIC +,1,1 - MS1 Match,MAM03124c,30817
356,Valine,Organic acids,Amino acids,6287,16414,HMDB0000883,NaN,C00183,KZSNJWFQEVHDMF-BYPYZUCNSA-N,RM0136018,...,Yes,Yes,Yes,Yes,"AFEKTA, AUTh, CEMBIO","CE+, GC, HILIC+",1,"1 - MS1 Match, 1 - MS2 Match",MAM03135c,16414


In [11]:
chebi_numbers = extract_chebi_numbers(data_file.CHEBI_ID_Step2.unique())

In [5]:
capitainer = data_file[data_file.Capitainer == "Yes"]
mitra = data_file[data_file.Mitra == "Yes"]
whatman = data_file[data_file.Whatman == "Yes"]
blood = data_file[data_file.Blood == "Yes"]
plasma = data_file[data_file.Plasma == "Yes"]

In [6]:
print("Capitainer annotations:", capitainer.CHEBI_ID_Step2.unique().shape[0])
print("Mitra annotations:", mitra.CHEBI_ID_Step2.unique().shape[0])
print("Whatman annotations:", whatman.CHEBI_ID_Step2.unique().shape[0])
print("Blood annotations:", blood.CHEBI_ID_Step2.unique().shape[0])
print("Plasma annotations:", plasma.CHEBI_ID_Step2.unique().shape[0])

Capitainer annotations: 335
Mitra annotations: 332
Whatman annotations: 331
Blood annotations: 290
Plasma annotations: 312


In [14]:
dict_human1 = parser.get_chebi_numbers()

In [33]:
df_human1 = parser.to_identifier_df()

In [51]:
df_human1

,chebi,kegg,metanetx,vmhmetabolite,hmdb,lipidmaps,pubchem
HUMAN1_ID,,,,,,,
MAM00001c,15389,C00964,MNXM45735,carveol,NaN,NaN,NaN
MAM00001e,15389,C00964,MNXM45735,carveol,NaN,NaN,NaN
MAM00002c,36740,C09880,MNXM163755,appnn,HMDB0006525,NaN,6654
MAM00002e,36740,C09880,MNXM163755,appnn,HMDB0006525,NaN,6654
MAM00003c,78990,NaN,MNXM150165,M00003,NaN,LMFA01030283,NaN
...,...,...,...,...,...,...,...
MAM01316n,60721,NaN,MNXM731626,NaN,NaN,NaN,NaN
MAM00767n,84503,NaN,MNXM733937,NaN,NaN,NaN,NaN
MAM01435m,16755,C02528,MNXM1183,HC00958,HMDB0000518,LMST04010032,10133


In [54]:
# Convert df_human1 CHEBI to integers for comparison
df_human1['chebi_int'] = pd.to_numeric(df_human1['chebi'], errors='coerce').astype('Int64')

# Create a mask for matching CHEBI numbers and get HUMAN1_IDs
mask = df_human1['chebi_int'].isin(chebi_numbers)
matched_human1_ids = df_human1[mask].index.tolist()

# Find CHEBI numbers that were NOT found in df_human1
not_found_chebi_ids = [
    chebi_id for chebi_id in chebi_numbers 
    if chebi_id not in df_human1['chebi_int'].values
]

print(f"Matched HUMAN1_IDs count: {len(matched_human1_ids)}")
print(f"Not found CHEBI IDs count: {len(not_found_chebi_ids)}")
print(f"\nMatched HUMAN1_IDs (first 10): {matched_human1_ids[:10]}")
print(f"Not found CHEBI IDs (first 10): {not_found_chebi_ids[:10]}")

# Store for later use
capitainer_results = {
    'matched_human1_ids': matched_human1_ids,
    'not_found_chebi_ids': not_found_chebi_ids
}

Matched HUMAN1_IDs count: 454
Not found CHEBI IDs count: 225

Matched HUMAN1_IDs (first 10): ['MAM00075x', 'MAM00248c', 'MAM00536c', 'MAM00536e', 'MAM00569c', 'MAM00648c', 'MAM00648e', 'MAM00674c', 'MAM00745c', 'MAM00745r']
Not found CHEBI IDs (first 10): [73037, 143745, 534, 16070, 190947, 16312, 156290, 86534, 28238, 16020]


In [55]:
# Debug: Check data types and sample values
print(f"Type of df_human1['chebi'] values: {type(df_human1['chebi'].iloc[0])}")
print(f"Type of chebi_numbers[0]: {type(chebi_numbers[0])}")
print(f"\nSample df_human1 CHEBI values: {df_human1['chebi'].dropna().unique()[:10]}")
print(f"Sample chebi_numbers values: {chebi_numbers[:10]}")

# Check if there's an overlap
df_chebi_set = set(df_human1['chebi'].dropna().values)
cap_chebi_set = set(chebi_numbers)
overlap = df_chebi_set & cap_chebi_set
print(f"\nOverlap found: {len(overlap)} common CHEBI IDs")
print(f"Overlap examples: {list(overlap)[:10]}")

Type of df_human1['chebi'] values: <class 'str'>
Type of chebi_numbers[0]: <class 'int'>

Sample df_human1 CHEBI values: ['15389' '36740' '78990' '34127' '74069' '77220' '76410' '53460' '74328'
 '15548']
Sample chebi_numbers values: [73037, 15725, 143745, 534, 16070, 190947, 16312, 156290, 86534, 28238]

Overlap found: 0 common CHEBI IDs
Overlap examples: []


In [56]:
not_found_chebi_ids

[73037,
 143745,
 534,
 16070,
 190947,
 16312,
 156290,
 86534,
 28238,
 16020,
 19062,
 27596,
 19065,
 35621,
 133223,
 231282,
 30845,
 19660,
 75454,
 69426,
 77761,
 75456,
 147319,
 39912,
 11901,
 37081,
 73739,
 28931,
 41254,
 37051,
 16831,
 65101,
 33070,
 43580,
 62208,
 145094,
 17050,
 178408,
 143078,
 61116,
 47069,
 86564,
 17597,
 17254,
 166563,
 17780,
 28871,
 87316,
 42989,
 28664,
 64294,
 232234,
 143265,
 73788,
 73757,
 43433,
 37742,
 183479,
 17475,
 73126,
 27732,
 135445,
 30776,
 28717,
 73051,
 77086,
 84634,
 73074,
 73024,
 84838,
 21949,
 73026,
 70819,
 165624,
 176464,
 84834,
 165628,
 16347,
 156179,
 68505,
 67042,
 67033,
 16113,
 16231,
 68641,
 17450,
 133094,
 133040,
 31459,
 28689,
 145234,
 28123,
 176392,
 4828,
 37655,
 741548,
 31575,
 193350,
 37714,
 37736,
 4139,
 168262,
 28393,
 17234,
 14314,
 232252,
 73804,
 17754,
 17687,
 166732,
 89929,
 73801,
 73922,
 17072,
 28775,
 190388,
 141437,
 232250,
 74058,
 74060,
 27747,
 52023

In [60]:
def has_id_level_one(value):
    if pd.isna(value):
        return False
    numbers = re.findall(r"\d+", str(value))
    return any(int(number) == 1 for number in numbers)

missing_chebi_ids_level_one = []
missing_chebi_rows_level_one = []

for chebi_id in not_found_chebi_ids:
    subset = data_file[data_file.CHEBI_ID_Step2 == f"CHEBI:{chebi_id}"]
    if subset.empty:
        continue

    level_one_rows = subset[subset["ID level"].apply(has_id_level_one)]
    if not level_one_rows.empty:
        missing_chebi_ids_level_one.append(chebi_id)
        missing_chebi_rows_level_one.append(level_one_rows)

print("Missing CHEBI IDs with ID level 1:")
print(missing_chebi_ids_level_one)

print("\nCount:", len(missing_chebi_ids_level_one))

Missing CHEBI IDs with ID level 1:
[73037, 16312, 86534, 28238, 16020, 19062, 27596, 19065, 35621, 133223, 30845, 19660, 69426, 77761, 11901, 37081, 73739, 37051, 16831, 65101, 178408, 143078, 86564, 17597, 17780, 28871, 42989, 28664, 64294, 232234, 143265, 73788, 43433, 27732, 30776, 28717, 73051, 77086, 84634, 73074, 73024, 84838, 21949, 73026, 70819, 165624, 176464, 84834, 165628, 16347, 16113, 16231, 68641, 133040, 145234, 28123, 4828, 741548, 37736, 4139, 168262, 28393, 17234, 14314, 232252, 73804, 17687, 166732, 89929, 73801, 73922, 17072, 28775, 232250, 27747, 52023, 16373, 43355, 19030, 73580, 61993, 17016, 232187, 45658, 136629, 136561, 21756, 133185, 27410, 27838, 165870, 21553, 40410, 176478, 63153, 17012, 21563, 44215, 15620, 191185, 64399, 90344, 15761, 70749, 84058, 30753, 46905, 25858, 233697, 232192, 72723, 28821, 35280, 39311, 52742, 9008, 21264, 16119, 28946, 28177, 18123, 232286, 167608]

Count: 123


In [66]:
# Extract KEGG IDs from data_file for missing CHEBIs with ID level 1
kegg_ids_from_missing = []

for chebi_id in missing_chebi_ids_level_one:
    subset = data_file[data_file.CHEBI_ID_Step2 == f"CHEBI:{chebi_id}"]
    kegg_values = subset["KEGG_ID"].dropna().unique()
    kegg_ids_from_missing.extend(kegg_values)

# Remove duplicates and convert to list
kegg_ids_from_missing = list(set(kegg_ids_from_missing))
kegg_ids_from_missing = sorted(kegg_ids_from_missing)

print(f"Total KEGG IDs extracted from missing CHEBIs: {len(kegg_ids_from_missing)}")
print(f"Sample KEGG IDs: {kegg_ids_from_missing[:10]}")

# Check for matches in df_human1['kegg']
df_human1_kegg = set(df_human1['kegg'].dropna().values)
kegg_ids_from_missing_set = set(kegg_ids_from_missing)

kegg_matching_in_human1 = sorted(
    kegg_ids_from_missing_set & df_human1_kegg
)
kegg_not_matching_in_human1 = sorted(
    kegg_ids_from_missing_set - df_human1_kegg
)

print(f"\nKEGG IDs matching in df_human1: {len(kegg_matching_in_human1)}")
print(f"KEGG IDs NOT matching in df_human1: {len(kegg_not_matching_in_human1)}")

print(f"\nMatching KEGG IDs: {kegg_matching_in_human1[:20]}")
print(f"Not matching KEGG IDs: {kegg_not_matching_in_human1[:20]}")

kegg_check_results = {
    'kegg_from_missing': kegg_ids_from_missing,
    'kegg_matching': kegg_matching_in_human1,
    'kegg_not_matching': kegg_not_matching_in_human1
}

Total KEGG IDs extracted from missing CHEBIs: 55
Sample KEGG IDs: ['-', 'C00003', 'C00029', 'C00031', 'C00092', 'C00117', 'C00124', 'C00187', 'C00329', 'C00334']

KEGG IDs matching in df_human1: 19
KEGG IDs NOT matching in df_human1: 36

Matching KEGG IDs: ['C00003', 'C00029', 'C00031', 'C00092', 'C00117', 'C00187', 'C00329', 'C00334', 'C00354', 'C00645', 'C01585', 'C01835', 'C01921', 'C02862', 'C03793', 'C05842', 'C05843', 'C15517', 'C19910']
Not matching KEGG IDs: ['-', 'C00124', 'C00487', 'C00493', 'C00568', 'C00633', 'C00643', 'C01004', 'C01035', 'C01152', 'C01546', 'C01657', 'C01924', 'C02242', 'C02494', 'C03139', 'C03299', 'C03440', 'C03882', 'C04545']


In [68]:
len(kegg_not_matching_in_human1)

36

In [70]:
# Map the 36 not-matching KEGG IDs back to their corresponding CHEBI IDs
chebi_ids_for_not_matching_kegg = {}

for kegg_id in kegg_not_matching_in_human1:
    chebi_list = data_file[data_file['KEGG_ID'] == kegg_id]['CHEBI_ID_Step2'].dropna().unique()
    if len(chebi_list) > 0:
        chebi_ids_for_not_matching_kegg[kegg_id] = list(chebi_list)

print(f"KEGG IDs with corresponding CHEBIs: {len(chebi_ids_for_not_matching_kegg)}")
print("\nMapping of not-matching KEGG IDs to CHEBI IDs:")
for kegg_id, chebi_ids in sorted(chebi_ids_for_not_matching_kegg.items()):
    print(f"{kegg_id}: {chebi_ids}")

# Flatten all CHEBI IDs for these non-matching KEGGs
all_chebi_for_not_matching_kegg = []
for chebi_list in chebi_ids_for_not_matching_kegg.values():
    all_chebi_for_not_matching_kegg.extend(chebi_list)

all_chebi_for_not_matching_kegg = sorted(list(set(all_chebi_for_not_matching_kegg)))

print(f"\nTotal unique CHEBI IDs for not-matching KEGGs: {len(all_chebi_for_not_matching_kegg)}")
print(f"CHEBI IDs: {all_chebi_for_not_matching_kegg}")

KEGG IDs with corresponding CHEBIs: 36

Mapping of not-matching KEGG IDs to CHEBI IDs:
-: ['CHEBI:73037', 'CHEBI:143745', 'CHEBI:534', 'CHEBI:190947', 'CHEBI:231282', 'CHEBI:19660', 'CHEBI:75454', 'CHEBI:75456', 'CHEBI:147319', 'CHEBI:11901', 'CHEBI:16831', 'CHEBI:62208', 'CHEBI:145094', 'CHEBI:143078', 'CHEBI:61116', 'CHEBI:47069', 'CHEBI:232234', 'CHEBI:37742', 'CHEBI:183479', 'CHEBI:73126', 'CHEBI:73024', 'CHEBI:16905', 'CHEBI:190388', 'CHEBI:189575', 'CHEBI:83047', 'CHEBI:73218', 'CHEBI:83054', 'CHEBI:17016', 'CHEBI:232187', 'CHEBI:136561', 'CHEBI:25682', 'CHEBI:177033', 'CHEBI:183936', 'CHEBI:191185', 'CHEBI:184082', 'CHEBI:70749', 'CHEBI:44605', 'CHEBI:46905', 'CHEBI:233009', 'CHEBI:73134', 'CHEBI:73127', 'CHEBI:232192', 'CHEBI:133880', 'CHEBI:232286', 'CHEBI:143353']
C00124: ['CHEBI:4139']
C00487: ['CHEBI:16347']
C00493: ['CHEBI:16119']
C00568: ['CHEBI:30753']
C00633: ['CHEBI:17597']
C00643: ['CHEBI:17780']
C01004: ['CHEBI:18123']
C01035: ['CHEBI:86564']
C01152: ['CHEBI:27596']


In [72]:
print(f"Total unique CHEBI IDs for the 36 not-matching KEGG IDs: {len(all_chebi_for_not_matching_kegg)}")
print(f"\nAll CHEBI IDs:")
print(all_chebi_for_not_matching_kegg)

Total unique CHEBI IDs for the 36 not-matching KEGG IDs: 80

All CHEBI IDs:
['CHEBI:11901', 'CHEBI:133880', 'CHEBI:136561', 'CHEBI:143078', 'CHEBI:143353', 'CHEBI:143745', 'CHEBI:145094', 'CHEBI:147319', 'CHEBI:15620', 'CHEBI:16020', 'CHEBI:16119', 'CHEBI:16231', 'CHEBI:16347', 'CHEBI:16373', 'CHEBI:16831', 'CHEBI:16905', 'CHEBI:17016', 'CHEBI:17072', 'CHEBI:17597', 'CHEBI:177033', 'CHEBI:17780', 'CHEBI:18123', 'CHEBI:183479', 'CHEBI:183936', 'CHEBI:184082', 'CHEBI:189575', 'CHEBI:19030', 'CHEBI:190388', 'CHEBI:19062', 'CHEBI:190947', 'CHEBI:191185', 'CHEBI:19660', 'CHEBI:21264', 'CHEBI:21563', 'CHEBI:231282', 'CHEBI:232187', 'CHEBI:232192', 'CHEBI:232234', 'CHEBI:232286', 'CHEBI:233009', 'CHEBI:25682', 'CHEBI:25858', 'CHEBI:27596', 'CHEBI:27732', 'CHEBI:27747', 'CHEBI:28123', 'CHEBI:28177', 'CHEBI:28238', 'CHEBI:28664', 'CHEBI:28717', 'CHEBI:28821', 'CHEBI:28871', 'CHEBI:28946', 'CHEBI:30753', 'CHEBI:30845', 'CHEBI:35280', 'CHEBI:37742', 'CHEBI:39311', 'CHEBI:4139', 'CHEBI:44605', 'CH

In [73]:
# Analyze the KEGG-to-CHEBI mapping to understand why 36 KEGGs map to 80 CHEBIs
print("KEGG-to-CHEBI Mapping Analysis:")
print("=" * 60)

# Count CHEBIs per KEGG
kegg_to_chebi_count = {}
for kegg_id, chebi_ids in chebi_ids_for_not_matching_kegg.items():
    kegg_to_chebi_count[kegg_id] = len(chebi_ids)

# Show distribution
from collections import Counter
distribution = Counter(kegg_to_chebi_count.values())

print(f"\nDistribution of CHEBI IDs per KEGG ID:")
for num_chebis in sorted(distribution.keys()):
    count = distribution[num_chebis]
    print(f"  {num_chebis} CHEBI(s) -> {count} KEGG ID(s)")

print(f"\nTotal KEGG IDs: {len(kegg_to_chebi_count)}")
print(f"Total unique CHEBI IDs: {len(all_chebi_for_not_matching_kegg)}")
print(f"Average CHEBIs per KEGG: {len(all_chebi_for_not_matching_kegg) / len(kegg_to_chebi_count):.2f}")

# Show examples
print("\nExamples:")
for kegg_id, num in sorted(kegg_to_chebi_count.items(), key=lambda x: -x[1])[:5]:
    chebi_ids = chebi_ids_for_not_matching_kegg[kegg_id]
    print(f"  {kegg_id} -> {len(chebi_ids)} CHEBIs: {chebi_ids}")

KEGG-to-CHEBI Mapping Analysis:

Distribution of CHEBI IDs per KEGG ID:
  1 CHEBI(s) -> 35 KEGG ID(s)
  45 CHEBI(s) -> 1 KEGG ID(s)

Total KEGG IDs: 36
Total unique CHEBI IDs: 80
Average CHEBIs per KEGG: 2.22

Examples:
  - -> 45 CHEBIs: ['CHEBI:73037', 'CHEBI:143745', 'CHEBI:534', 'CHEBI:190947', 'CHEBI:231282', 'CHEBI:19660', 'CHEBI:75454', 'CHEBI:75456', 'CHEBI:147319', 'CHEBI:11901', 'CHEBI:16831', 'CHEBI:62208', 'CHEBI:145094', 'CHEBI:143078', 'CHEBI:61116', 'CHEBI:47069', 'CHEBI:232234', 'CHEBI:37742', 'CHEBI:183479', 'CHEBI:73126', 'CHEBI:73024', 'CHEBI:16905', 'CHEBI:190388', 'CHEBI:189575', 'CHEBI:83047', 'CHEBI:73218', 'CHEBI:83054', 'CHEBI:17016', 'CHEBI:232187', 'CHEBI:136561', 'CHEBI:25682', 'CHEBI:177033', 'CHEBI:183936', 'CHEBI:191185', 'CHEBI:184082', 'CHEBI:70749', 'CHEBI:44605', 'CHEBI:46905', 'CHEBI:233009', 'CHEBI:73134', 'CHEBI:73127', 'CHEBI:232192', 'CHEBI:133880', 'CHEBI:232286', 'CHEBI:143353']
  C00124 -> 1 CHEBIs: ['CHEBI:4139']
  C00487 -> 1 CHEBIs: ['CHEBI: